# Rideshare fare model
Routes every tract centroid to each airport terminal, prices the trip with the published rate card, and samples tracts to collect real Uber/Lyft quotes for the regression.

In [ ]:
import os

import geopandas as gpd
import pandas as pd

from src.config import PROCESSED_DIR
from src.routing import PBF_URL, load_graph, tract_airport_routes
from src.fares import rate_card_fare, quote_sample, fit_fare_model, predict_fares

### Road network
Download the Geofabrik Northern California extract (~625 MB) to `PBF` first, e.g. `curl -L -o <PBF> <PBF_URL>`. The first run filters it to major roads and caches the graph next to it (~1.5 min).

In [ ]:
PBF = os.path.expanduser("~/Library/Caches/airport_transit/norcal-latest.osm.pbf")
print(PBF_URL)
centroids = gpd.read_file(f"{PROCESSED_DIR}/tract_centroids_4326.gpkg")
G = load_graph(centroids, PBF)
print(f"nodes: {len(G):,}  edges: {G.number_of_edges():,}")

### Tract to airport routes
Fastest free-flow route; tolls are charged when a route crosses a bridge in its tolled direction.

In [ ]:
routes = tract_airport_routes(G, centroids)
routes.groupby("code")[["miles", "minutes", "toll"]].describe().round(1).T

### Rate-card baseline
What the published rate card implies, before surge and upfront-pricing adjustments.

In [ ]:
routes = rate_card_fare(routes)
routes.groupby("code")["rate_card_fare"].describe().round(2)

### Quote sample
Written once to `fare_quotes.csv` so filled-in quotes are never overwritten. For each row, open the app, enter the lat/lon as pickup and the airport as drop-off, and record `service`, `requested_at`, `price` and `quoted_minutes`. Collect all quotes in the same time window (e.g. weekday 10am-2pm) or record it so time of day can be a predictor.

In [ ]:
quotes_path = f"{PROCESSED_DIR}/fare_quotes.csv"
if not os.path.exists(quotes_path):
    workers = pd.read_csv(f"{PROCESSED_DIR}/airport_workers_by_tract.csv", dtype={"GEOID": str})
    quote_sample(routes, workers, centroids).to_csv(quotes_path, index=False)
quotes = pd.read_csv(quotes_path, dtype={"GEOID": str})
print(f"{quotes['price'].notna().sum()} of {len(quotes)} quotes collected")

### Regression
Runs once quotes are filled in.

In [ ]:
if quotes["price"].notna().sum() >= 20:
    model = fit_fare_model(quotes)
    print(model.summary())
    fares = predict_fares(model, routes)
else:
    print("collect at least 20 quotes first")